In [1]:
from pathlib import Path

import numpy as np
import pyvista as pv

from cardiac_electrophysiology.components import transform
from cardiac_electrophysiology.utils import analysis, mesh_utils, visualization

In [2]:
data_path = Path("../data/")
mesh_path = data_path / "mesh.vtu"
basis_vecs_path = data_path / "basis_vecs.npy"
fiber_field_path = data_path / "fiber_field.npy"
fiber_field_ensemble_path = data_path / "fiber_fields_mapped_to_target.npz"

In [3]:
mesh = pv.read(mesh_path)
basis_vecs = np.load(basis_vecs_path)
fiber_field = np.load(fiber_field_path)
fiber_field_ensemble = np.load(fiber_field_ensemble_path)
simplex_to_vertex_matrix = mesh_utils.assemble_simplex_to_vertex_interpolation_matrix(
    mesh.cells.reshape(-1, 4)[:, 1:], mesh.points
)

In [4]:
angle_transformator = transform.AngleFiberTransformator(basis_vecs[..., 0], basis_vecs[..., 1])
ground_truth_angle_field = angle_transformator.compute_angle_from_fiber(fiber_field)
ground_truth_angle_field = simplex_to_vertex_matrix @ ground_truth_angle_field
ground_truth_angle_field = analysis.shift_angles_to_minimize_axial_variance(
    ground_truth_angle_field, axis=1
).flatten()

angle_field_ensemble = []
for field in fiber_field_ensemble.values():
    angle_field = angle_transformator.compute_angle_from_fiber(field)
    angle_field = simplex_to_vertex_matrix @ angle_field
    angle_field_ensemble.append(angle_field)

mean_angle_field, var_angle_field = analysis.compute_axial_mean_and_variance(
    np.array(angle_field_ensemble), axis=0
)
mean_angle_field = analysis.shift_angles_to_minimize_axial_variance(
    mean_angle_field, axis=1
).flatten()
mean_angle_field_const = np.ones_like(mean_angle_field) * np.mean(mean_angle_field)
np.save(data_path / "ground_truth_angle_field.npy", ground_truth_angle_field)
np.save(data_path / "mean_angle_field.npy", mean_angle_field)
np.save(data_path / "mean_angle_field_const.npy", mean_angle_field_const)
np.save(data_path / "var_angle_field.npy", var_angle_field)

In [8]:
diff_field = analysis.compute_axial_data_diff(ground_truth_angle_field, mean_angle_field)
visualization.visualize_vector_field(mesh, fiber_field, scaling_factor=0.5)
visualization.visualize_scalar_field(mesh, ground_truth_angle_field, circular=True)
visualization.visualize_scalar_field(mesh, mean_angle_field, circular=True)
visualization.visualize_scalar_field(mesh, diff_field, circular=True)
visualization.visualize_scalar_field(mesh, var_angle_field, circular=False)

Widget(value='<iframe src="http://localhost:39923/index.html?ui=P_0x7fddb251a710_9&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:39923/index.html?ui=P_0x7fde087a42d0_10&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:39923/index.html?ui=P_0x7fddb251b890_11&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:39923/index.html?ui=P_0x7fddb251b750_12&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:39923/index.html?ui=P_0x7fddbc4c8910_13&reconnect=auto" class="pyv…